[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S12_pandas_combinar_fechas.ipynb)

# Sesión 12 · pandas: combinar tablas y trabajar con fechas

**Módulo 3: Pandas** · ⏱️ Duración estimada: 60 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Apilar tablas con `concat` y unirlas por una columna con `merge`.
2. Elegir el tipo de unión (`left`, `inner`, `outer`) y detectar filas sin pareja.
3. Convertir textos en fechas con `to_datetime`, extraer partes con `.dt` y resumir por periodo con `resample`.
4. Encadenar operaciones de forma legible (*method chaining*).

## 📋 Qué debes saber antes
Sesiones 9 a 11: seleccionar, limpiar y agrupar DataFrames.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.
- Después de cada `merge`, compara la cantidad de filas antes y después: es la forma más rápida de detectar un error.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Crea las tablas de práctica y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión y las funciones que revisan tus respuestas.
import copy
import datetime as dt
import hashlib
import math
import statistics
from collections import defaultdict

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---------- Datos de práctica: ventas de tiendas ----------
productos = pd.DataFrame({
    "producto": ["polo básico", "jean slim", "casaca denim", "gorra", "medias"],
    "categoria": ["polo", "jean", "casaca", "accesorio", "accesorio"],
    "costo_unitario": [18.0, 65.0, 95.0, 9.0, 4.5],
})
_PRECIOS = {"polo básico": 39.9, "jean slim": 129.9, "casaca denim": 189.9, "gorra": 25.0, "medias": 12.9, "bufanda": 45.0}
clientes = pd.DataFrame({
    "cliente_id": [f"C{k:02d}" for k in range(1, 13)],
    "distrito": rng.choice(["Miraflores", "Surco", "Lince", "Barranco"], 12),
    "segmento": rng.choice(["nuevo", "frecuente", "vip"], 12, p=[0.4, 0.4, 0.2]),
})


def _mes_ventas(mes, n):
    filas = []
    for _ in range(n):
        cli = f"C{int(rng.integers(1, 11)):02d}" if rng.random() > 0.08 else "C99"      # C99 no está en `clientes`
        prod = str(rng.choice(list(_PRECIOS), p=[0.25, 0.2, 0.15, 0.15, 0.15, 0.1]))  # la bufanda no está en `productos`
        u = int(rng.integers(1, 5))
        filas.append([f"2026-{mes:02d}-{int(rng.integers(1, 29)):02d}", cli, prod, u, round(u * _PRECIOS[prod], 2)])
    return pd.DataFrame(filas, columns=["fecha", "cliente_id", "producto", "unidades", "importe"])


ventas_ene, ventas_feb, ventas_mar = _mes_ventas(1, 22), _mes_ventas(2, 18), _mes_ventas(3, 25)

# ---------- Fechas escritas como día/mes/año, con errores ----------
_f = [f"{int(rng.integers(1, 29)):02d}/{int(rng.integers(1, 7)):02d}/2026" for _ in range(20)]
_f[6], _f[13] = "31/02/2026", "sin fecha"
movs_fechas = pd.DataFrame({"fecha": _f,
                            "concepto": rng.choice(["supermercado", "restaurante", "servicios", "cajero"], 20),
                            "monto": np.round(-rng.uniform(15, 400, 20), 2)})

# ---------- Una serie diaria con un mes sin datos (marzo) ----------
_dias = [dt.date(2026, 1, 1) + dt.timedelta(days=k) for k in range(120)]
_dias = [d for d in _dias if d.month != 3]
diario = pd.DataFrame({"fecha": [d.isoformat() for d in _dias], "monto": np.round(rng.normal(1500, 400, len(_dias)), 2)})

# ---------- Datos de práctica: movimientos bancarios del primer trimestre ----------
clientes_banco = pd.DataFrame({"cliente_id": [f"B{k:02d}" for k in range(1, 16)],
                               "segmento": rng.choice(["clásico", "preferente", "premium"], 15, p=[0.5, 0.3, 0.2])})


def _mes_movs(mes, n):
    filas = []
    for _ in range(n):
        cli = f"B{int(rng.integers(1, 13)):02d}" if rng.random() > 0.1 else "B99"
        tipo = str(rng.choice(["deposito", "retiro", "pago"], p=[0.3, 0.4, 0.3]))
        monto = rng.uniform(300, 4000) if tipo == "deposito" else -rng.uniform(20, 1500)
        filas.append([f"2026-{mes:02d}-{int(rng.integers(1, 29)):02d}", cli, tipo, round(float(monto), 2)])
    return pd.DataFrame(filas, columns=["fecha", "cliente_id", "tipo", "monto"])


movs_ene, movs_feb, movs_mar = _mes_movs(1, 20), _mes_movs(2, 20), _mes_movs(3, 20)
_q1 = [dt.date(2026, 1, 1) + dt.timedelta(days=k) for k in range(90)]
tipos_cambio = pd.DataFrame({"fecha": [d.isoformat() for d in _q1], "tc": np.round(rng.uniform(3.68, 3.82, 90), 3)})

_NOMBRES = ["productos", "clientes", "ventas_ene", "ventas_feb", "ventas_mar", "movs_fechas", "diario",
            "clientes_banco", "movs_ene", "movs_feb", "movs_mar", "tipos_cambio"]
_D = copy.deepcopy({k: globals()[k] for k in _NOMBRES})
# Versiones en listas de Python: los verificadores recalculan con bucles y diccionarios, sin pandas.
_R = {k: [dict(zip(v.columns, map(lambda x: x.item() if hasattr(x, "item") else x, fila))) for fila in v.itertuples(index=False)]
      for k, v in _D.items()}

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")


def _ts(d):
    """Texto con el que pandas muestra un Timestamp a medianoche."""
    return None if d is None else str(dt.datetime(d.year, d.month, d.day))


def _clave(fila):
    return tuple("~" if x is None else (round(x, 4) if isinstance(x, float) else x) for x in map(_norm, fila))


def _df_sin_orden(r, nombre, columnas, filas, pista):
    """Como _df, pero sin exigir un orden de filas (útil para merge inner y outer)."""
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    if [str(c) for c in v.columns] != columnas:
        r.mal(f"`{nombre}` debería tener las columnas {columnas}, en ese orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if sorted(map(_clave, v.itertuples(index=False))) == sorted(map(_clave, filas)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus filas no coinciden; {pista}.")


def _trimestre():
    return _R["ventas_ene"] + _R["ventas_feb"] + _R["ventas_mar"]


_COLS_T = ["fecha", "cliente_id", "producto", "unidades", "importe"]


def _unir(izq, der, clave, como):
    """merge de referencia hecho con diccionarios."""
    cols_der = [c for c in der[0] if c != clave]
    indice = defaultdict(list)
    for f in der:
        indice[f[clave]].append(f)
    salida, usadas = [], set()
    for f in izq:
        pares = indice.get(f[clave], [])
        if pares:
            usadas.add(f[clave])
            salida += [[f[c] for c in f] + [g[c] for c in cols_der] for g in pares]
        elif como in ("left", "outer"):
            salida.append([f[c] for c in f] + [None] * len(cols_der))
    if como == "outer":
        cols_izq = list(izq[0])
        for g in der:
            if g[clave] not in usadas:
                salida.append([g[clave] if c == clave else None for c in cols_izq] + [g[c] for c in cols_der])
    return salida


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    t = _trimestre()
    _df(r, "trimestre", _COLS_T, [list(f.values()) for f in t], "enero, luego febrero y luego marzo, con un índice nuevo de 0 en adelante",
        indice=list(range(len(t))))
    x = r.var("trimestre_con_origen")
    if x is not _FALTA:
        esperado = ["ene"] * len(_R["ventas_ene"]) + ["feb"] * len(_R["ventas_feb"]) + ["mar"] * len(_R["ventas_mar"])
        if isinstance(x, pd.DataFrame) and x.index.nlevels == 2 and list(x.index.get_level_values(0)) == esperado:
            r.ok("`trimestre_con_origen` es correcto.")
        else:
            r.mal("`trimestre_con_origen` debería tener un índice de dos niveles cuyo primer nivel dice \"ene\", \"feb\" o \"mar\".")
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_indice_unico": "98f1c15b97f0aab4934b54ee33b6a9fd148f07ee171d3618e796e5ba5b2cbf3d",
        "pred_forma_distintas": "c67fa4c7517a371b1d86c45a58fb1e30b40c3dd192b55ca9719f4bc55f1656f9",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    t, c, p = _trimestre(), _R["clientes"], _R["productos"]
    cols = _COLS_T + ["distrito", "segmento"]
    _df(r, "con_cliente", cols, _unir(t, c, "cliente_id", "left"), "todas las ventas, con los datos del cliente cuando existen")
    _df_sin_orden(r, "solo_conocidos", cols, _unir(t, c, "cliente_id", "inner"), "solo las ventas de clientes que están en `clientes`")
    _df_sin_orden(r, "todos", cols, _unir(t, c, "cliente_id", "outer"), "todas las ventas y todos los clientes, aunque no coincidan")
    compradores = {f["cliente_id"] for f in t}
    r.valor("clientes_sin_compras", sorted(f["cliente_id"] for f in c if f["cliente_id"] not in compradores), list,
            "lista ordenada de los clientes de `clientes` que no aparecen en `trimestre`")
    con_costo = _unir(t, p, "producto", "left")
    _df(r, "con_costo", _COLS_T + ["categoria", "costo_unitario"], con_costo, "todas las ventas, con la categoría y el costo del producto")
    r.valor("productos_sin_costo", sorted({f[2] for f in con_costo if f[-1] is None}), list,
            "lista ordenada de los productos vendidos que no están en `productos`")
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_outer": "4bfc84ac8fd1e690b2309147b1a4cddd6d7a2714fa6aa501a4f3d95ced8e3158",
        "pred_inner": "d024f6472cd1381eeddb28a59daaff16528770c069c615713e06c20d6b4498fd",
        "pred_duplicada": "b4d4f68c2268549cada66d24ae3a1902d4152a98ad614bbf891edf235ef99218",
    })
    r.fin()


def _fecha_dma(texto):
    try:
        d, m, a = map(int, texto.split("/"))
        return dt.date(a, m, d)
    except ValueError:
        return None


def check_ejercicio_3():
    r = _Revision("Ejercicio 3 · Parte A")
    filas, desde = [], []
    fechas = [_fecha_dma(f["fecha"]) for f in _R["movs_fechas"]]
    inicio = min(d for d in fechas if d)
    for f, d in zip(_R["movs_fechas"], fechas):
        filas.append([_ts(d), f["concepto"], f["monto"], d.month if d else None, d.weekday() if d else None,
                      bool(d and d.weekday() >= 5)])
        desde.append((d - inicio).days if d else None)
    _df(r, "movs_f", ["fecha", "concepto", "monto", "mes", "dia_semana", "fin_de_semana"], filas,
        "revisa el formato día/mes/año, `errors=\"coerce\"` y las tres columnas nuevas")
    _esc(r, "n_fechas_invalidas", sum(d is None for d in fechas), "cuenta las fechas que no se pudieron convertir")
    _ser(r, "dias_desde_inicio", desde, "días transcurridos desde la fecha más antigua (NaN si la fecha no es válida)")
    _sin_cambios_df(r, "movs_fechas")
    r.fin()
    r = _Revision("Ejercicio 3 · Parte B")
    r.predicciones({
        "pred_lunes": "25af31a57c2047d854d189042b0ecfb66843c4c19d3dd9ce1403e0be3826a240",
        "pred_dias_feb": "cd0c612cbd1167c3be42cceb7d5dbb36f2ce763cb3afd0e87036f06d9b7d9bbb",
    })
    r.fin()


def _fin_de_mes(d):
    siguiente = dt.date(d.year + (d.month == 12), d.month % 12 + 1, 1)
    return siguiente - dt.timedelta(days=1)


def _periodos(pares, clave_fin, paso):
    grupos = defaultdict(list)
    for d, x in pares:
        grupos[clave_fin(d)].append(x)
    fines, actual = [], min(grupos)
    while actual <= max(grupos):
        fines.append(actual)
        actual = paso(actual)
    return fines, grupos


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    pares = [(dt.date.fromisoformat(f["fecha"]), f["monto"]) for f in _R["diario"]]
    _ser(r, "serie", [x for _, x in pares], "la columna monto con las fechas (convertidas) como índice", indice=[_ts(d) for d, _ in pares])
    fines, g = _periodos(pares, _fin_de_mes, lambda d: _fin_de_mes(d + dt.timedelta(days=1)))
    etiquetas = [_ts(d) for d in fines]
    _ser(r, "mensual", [math.fsum(g.get(d, [])) for d in fines], "la suma de cada mes (un mes sin datos suma 0)", indice=etiquetas, tol=1e-6)
    _ser(r, "mensual_prom", [round(statistics.fmean(g[d]), 2) if d in g else None for d in fines],
         "el promedio de cada mes, con 2 decimales", indice=etiquetas, tol=0.0051)
    fines, g = _periodos(pares, lambda d: d + dt.timedelta(days=6 - d.weekday()), lambda d: d + dt.timedelta(days=7))
    _ser(r, "semanal_max", [max(g[d]) if d in g else None for d in fines], "el máximo de cada semana (las semanas terminan en domingo)",
         indice=[_ts(d) for d in fines])
    _sin_cambios_df(r, "diario")
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_mes_vacio": "25af31a57c2047d854d189042b0ecfb66843c4c19d3dd9ce1403e0be3826a240",
    })
    r.fin()


def check_ejercicio_5():
    r = _Revision("Ejercicio 5")
    cat = {f["producto"]: (f["categoria"], f["costo_unitario"]) for f in _R["productos"]}
    tot = defaultdict(lambda: [0.0, 0.0])
    for f in _trimestre():
        if f["producto"] in cat:
            c, costo = cat[f["producto"]]
            tot[c][0] += f["importe"]
            tot[c][1] += f["importe"] - f["unidades"] * costo
    orden = sorted(tot, key=lambda c: -tot[c][1])
    _df(r, "reporte", ["total_importe", "total_margen", "margen_pct"],
        [[tot[c][0], tot[c][1], round(tot[c][1] * 100 / tot[c][0], 1)] for c in orden],
        "una fila por categoría, ordenadas de mayor a menor margen total", indice=orden, tol=0.051)
    r.fin()


def _reto_ref():
    seg = {f["cliente_id"]: f["segmento"] for f in _R["clientes_banco"]}
    tc = {f["fecha"]: f["tc"] for f in _R["tipos_cambio"]}
    filas = []
    for f in _R["movs_ene"] + _R["movs_feb"] + _R["movs_mar"]:
        d = dt.date.fromisoformat(f["fecha"])
        filas.append([d, f["cliente_id"], f["tipo"], f["monto"], seg.get(f["cliente_id"], "sin registro"), tc[f["fecha"]],
                      round(f["monto"] / tc[f["fecha"]], 2), d.month])
    return filas


def check_reto():
    r = _Revision("Reto final")
    filas = _reto_ref()
    _df(r, "movs", ["fecha", "cliente_id", "tipo", "monto", "segmento", "tc", "monto_usd", "mes"],
        [[_ts(f[0])] + f[1:] for f in filas], "revisa cada paso y el orden de las columnas nuevas", tol=0.0051)
    g = defaultdict(float)
    for f in filas:
        g[(f[4], f[7])] += f[3]
    claves = sorted(g)
    _ser(r, "flujo_segmento_mes", [round(g[k], 2) for k in claves], "suma del monto por segmento y mes, con 2 decimales",
         indice=claves, tol=0.0051)
    fines, gs = _periodos([(f[0], f[3]) for f in filas], lambda d: d + dt.timedelta(days=6 - d.weekday()), lambda d: d + dt.timedelta(days=7))
    _ser(r, "flujo_semanal", [round(math.fsum(gs.get(d, [])), 2) for d in fines], "suma del monto por semana, con 2 decimales",
         indice=[_ts(d) for d in fines], tol=0.0051)
    activos = {f[1] for f in filas}
    r.valor("clientes_inactivos", sorted(f["cliente_id"] for f in _R["clientes_banco"] if f["cliente_id"] not in activos), list,
            "lista ordenada de los clientes de `clientes_banco` sin movimientos")
    _esc(r, "n_sin_registro", sum(f[4] == "sin registro" for f in filas), "cuenta los movimientos de clientes que no están en `clientes_banco`")
    _sin_cambios_df(r, "movs_ene", "movs_feb", "movs_mar", "clientes_banco", "tipos_cambio")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    unidas = _unir(_trimestre(), _R["clientes"], "cliente_id", "outer")
    compradores = {f["cliente_id"] for f in _trimestre()}
    conocidos = {f["cliente_id"] for f in _R["clientes"]}
    conteo = {"both": sum(1 for f in unidas if f[0] is not None and f[1] in conocidos),
              "left_only": sum(1 for f in unidas if f[1] not in conocidos),
              "right_only": sum(1 for f in unidas if f[0] is None)}
    x = r.var("origen")
    if x is not _FALTA:
        if isinstance(x, pd.Series) and {str(k): int(v) for k, v in x.items()} == conteo:
            r.ok("`origen` es correcto.")
        else:
            r.mal("`origen` debería contar cuántas filas del merge outer vienen de ambas tablas, solo de la izquierda o solo de la derecha.")
    dias = defaultdict(float)
    for f in _trimestre():
        dias[dt.date.fromisoformat(f["fecha"]).weekday()] += f["importe"]
    _ser(r, "ventas_dia_semana", [dias[k] for k in sorted(dias)], "suma del importe por día de la semana (0 es lunes)", indice=sorted(dias))
    r.fin()


print("✅ Setup listo. Datos generados y verificadores cargados.")

### 📦 Tus datos de hoy
- `ventas_ene`, `ventas_feb`, `ventas_mar`: las ventas de cada mes, con las mismas columnas.
- `clientes` y `productos`: tablas de referencia. Ojo: hay ventas de un cliente y de un producto que no están en esas tablas.
- `movs_fechas`: movimientos con la fecha escrita como día/mes/año, con algunos errores.
- `diario`: un monto por día entre enero y abril, sin datos en marzo.
- Para el reto: `clientes_banco`, `movs_ene`, `movs_feb`, `movs_mar` y `tipos_cambio`.

In [ ]:
for nombre in ["ventas_ene", "clientes", "productos", "movs_fechas", "diario", "clientes_banco", "movs_ene", "tipos_cambio"]:
    tabla = globals()[nombre]
    print(f"--- {nombre} {tabla.shape}")
    print(tabla.head(3), "\n")

---
## 1. Apilar tablas: `concat`

### 📘 Concepto
`pd.concat([df1, df2, df3])` pone las tablas una debajo de otra. Es lo que necesitas cuando los datos llegan en un archivo por mes o por año.
- Las columnas se alinean **por nombre**. Si una tabla tiene una columna que otra no, esa columna queda con `NaN` en las filas que no la tenían.
- Por defecto se conservan los índices originales, así que pueden repetirse. Con `ignore_index=True` se crea un índice nuevo de 0 en adelante.
- Con `keys=["a", "b"]` se agrega un nivel de índice que dice de qué tabla viene cada fila.

In [ ]:
lunes_ej = pd.DataFrame({"tienda": ["Surco", "Lince"], "monto": [120.0, 80.0]})
martes_ej = pd.DataFrame({"tienda": ["Surco"], "monto": [95.0]})
print(pd.concat([lunes_ej, martes_ej]))                      # índices 0, 1, 0
print(pd.concat([lunes_ej, martes_ej], ignore_index=True))   # índices 0, 1, 2
print(pd.concat([lunes_ej, martes_ej], keys=["lun", "mar"]))

### ✍️ Tu turno · Ejercicio 1: el trimestre en una tabla
**Parte A.**
1. `trimestre`: las ventas de enero, febrero y marzo, en ese orden, con un índice nuevo de 0 en adelante.
2. `trimestre_con_origen`: las mismas tres tablas, pero con un nivel de índice que diga `"ene"`, `"feb"` o `"mar"`.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta | Formato |
|---|---|---|
| `pred_indice_unico` | `pd.concat([ventas_ene, ventas_feb]).index.is_unique` | `True` o `False` |
| `pred_forma_distintas` | `pd.concat([pd.DataFrame({"a": [1]}), pd.DataFrame({"b": [2]})]).shape` | tupla |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

`concat` recibe **una lista** de DataFrames.
</details>

<details><summary>💡 Pista 2</summary>

`trimestre` usa `ignore_index=True`; `trimestre_con_origen` usa `keys=["ene", "feb", "mar"]`.
</details>

---
## 2. Unir por una columna: `merge`

### 📘 Concepto
`izq.merge(der, on="clave", how=...)` junta columnas de dos tablas emparejando las filas que tienen el mismo valor en `clave`, como un BUSCARV.

| `how` | Qué filas quedan |
|---|---|
| `"left"` | todas las de la izquierda; si no tienen pareja, las columnas nuevas quedan en `NaN` |
| `"inner"` | solo las que tienen pareja en las dos tablas |
| `"outer"` | todas las de las dos tablas |
| `"right"` | todas las de la derecha |

⚠️ Si la clave se repite en la tabla derecha, cada fila de la izquierda se **duplica** una vez por pareja. Por eso conviene contar las filas antes y después del `merge`.

Para saber qué valores de una tabla no están en otra: `df[~df["clave"].isin(otra["clave"])]`.

In [ ]:
pedidos_ej = pd.DataFrame({"cliente": ["A", "B", "Z"], "monto": [100, 50, 70]})
datos_ej = pd.DataFrame({"cliente": ["A", "B", "C"], "ciudad": ["Lima", "Cusco", "Piura"]})
print(pedidos_ej.merge(datos_ej, on="cliente", how="left"))
print(pedidos_ej.merge(datos_ej, on="cliente", how="inner"))
print(pedidos_ej.merge(datos_ej, on="cliente", how="outer"))
print(datos_ej[~datos_ej["cliente"].isin(pedidos_ej["cliente"])])   # clientes sin pedidos

### ✍️ Tu turno · Ejercicio 2: ventas con datos de clientes y productos
**Parte A.** Con `trimestre`:
1. `con_cliente`: todas las ventas con el distrito y el segmento del cliente (`how="left"`).
2. `solo_conocidos`: solo las ventas de clientes que están en `clientes`.
3. `todos`: todas las ventas y todos los clientes, tengan o no pareja.
4. `clientes_sin_compras`: una **lista ordenada** con los `cliente_id` de `clientes` que no compraron en el trimestre.
5. `con_costo`: todas las ventas con la categoría y el costo unitario de `productos`, y `productos_sin_costo`: una lista ordenada con los productos vendidos que no están en `productos`.

**Parte B.** Predice **sin ejecutar**, con `A = pd.DataFrame({"k": [1, 2]})` y `B = pd.DataFrame({"k": [2, 3]})`:

| Variable | Pregunta |
|---|---|
| `pred_outer` | `len(A.merge(B, on="k", how="outer"))` |
| `pred_inner` | `len(A.merge(B, on="k", how="inner"))` |
| `pred_duplicada` | `len(pd.DataFrame({"k": [1]}).merge(pd.DataFrame({"k": [1, 1]}), on="k"))` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Los tres primeros puntos son el mismo `merge` con distinto `how`. Compara el largo de cada resultado con el de `trimestre`.
</details>

<details><summary>💡 Pista 2</summary>

Para `clientes_sin_compras`: filtra `clientes` con `~...isin(trimestre["cliente_id"])`, toma la columna y ordénala con `sorted(...)`. Para `productos_sin_costo`, busca en `con_costo` las filas con `costo_unitario` nulo.
</details>

---
## 3. Fechas: `to_datetime` y `.dt`

### 📘 Concepto
Una fecha guardada como texto no se puede restar ni agrupar por mes. `pd.to_datetime` la convierte en un tipo fecha (`datetime64`):
- `format="%d/%m/%Y"` indica el formato exacto (`%d` día, `%m` mes, `%Y` año de 4 cifras). Así "03/04/2026" se lee como 3 de abril y no como 4 de marzo.
- `errors="coerce"` convierte en `NaT` (el "nulo" de las fechas) lo que no se puede leer, en lugar de detenerse con un error.

Con el accesor **`.dt`** sacas partes de una columna de fechas: `.dt.year`, `.dt.month`, `.dt.day` y `.dt.dayofweek` (0 es lunes y 6 es domingo). Restar dos fechas da una duración; `.dt.days` la convierte en número de días.

In [ ]:
fechas_ej = pd.Series(["03/04/2026", "15/04/2026", "30/02/2026"])
convertidas_ej = pd.to_datetime(fechas_ej, format="%d/%m/%Y", errors="coerce")
print(convertidas_ej)
print(convertidas_ej.dt.month, convertidas_ej.dt.dayofweek)
print((convertidas_ej - convertidas_ej.min()).dt.days)

### ✍️ Tu turno · Ejercicio 3: fechas de movimientos
**Parte A.**
1. `movs_f`: una copia de `movs_fechas` en la que la columna fecha esté convertida a fecha (formato día/mes/año; lo que no se pueda leer, como nulo).
2. `n_fechas_invalidas`: cuántas fechas no se pudieron convertir.
3. Agrega a `movs_f`, en este orden, `mes`, `dia_semana` (0 es lunes) y `fin_de_semana` (`True` si es sábado o domingo).
4. `dias_desde_inicio`: cuántos días pasaron desde la fecha más antigua hasta cada fecha.

`movs_fechas` no debe cambiar.

**Parte B.** Predice **sin ejecutar** (la fecha de hoy, 25 de setiembre de 2026, es viernes):

| Variable | Pregunta |
|---|---|
| `pred_lunes` | `pd.Timestamp("2026-09-28").dayofweek` |
| `pred_dias_feb` | `(pd.Timestamp("2026-03-01") - pd.Timestamp("2026-02-01")).days` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Empieza con `movs_f = movs_fechas.copy()` y reasigna la columna fecha con `pd.to_datetime(...)`. Cuenta los nulos con `.isna().sum()`.
</details>

<details><summary>💡 Pista 2</summary>

Las columnas nuevas salen de `movs_f["fecha"].dt....`. `fin_de_semana` es una comparación sobre `dia_semana`.
</details>

---
## 4. Resumir por periodo: `resample`

### 📘 Concepto
Con las fechas en el **índice**, `resample` agrupa por periodos de calendario:

```python
serie.resample("ME").sum()    # por mes ("ME" = fin de mes)
serie.resample("W").max()     # por semana (terminan en domingo)
serie.resample("D").mean()    # por día
```

La etiqueta de cada periodo es su último día. Los periodos sin datos **también aparecen**: con `sum` valen 0 y con `mean` o `max` quedan como `NaN`.

Para poner la fecha en el índice: `df.set_index("fecha")`.

In [ ]:
cobros_ej = pd.DataFrame({"fecha": pd.to_datetime(["2026-01-05", "2026-01-20", "2026-03-02"]),
                          "monto": [100.0, 50.0, 80.0]})
serie_ej = cobros_ej.set_index("fecha")["monto"]
print(serie_ej.resample("ME").sum())     # febrero aparece con 0
print(serie_ej.resample("ME").mean())    # febrero aparece como NaN

### ✍️ Tu turno · Ejercicio 4: el flujo diario por mes y por semana
**Parte A.**
1. `serie`: la columna monto de `diario`, con las fechas convertidas como índice. Trabaja sobre una copia de `diario`.
2. `mensual`: la suma de cada mes.
3. `mensual_prom`: el promedio de cada mes, con 2 decimales.
4. `semanal_max`: el máximo de cada semana.

**Parte B.** Predice **sin ejecutar**: `pred_mes_vacio` es el valor de `mensual` en marzo (un número o `"nan"`).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

Copia `diario`, convierte su columna fecha con `pd.to_datetime`, pon la fecha como índice y toma la columna monto.
</details>

<details><summary>💡 Pista 2</summary>

Los tres resúmenes empiezan con `serie.resample(...)`: `"ME"` para meses y `"W"` para semanas.
</details>

---
## 5. Encadenar métodos (*method chaining*)

### 📘 Concepto
Muchas operaciones de pandas devuelven un DataFrame nuevo, así que se pueden encadenar en una sola expresión, un paso por línea, dentro de paréntesis:

```python
resultado = (
    df
    .merge(otra, on="clave")
    .assign(nueva=lambda d: d["a"] - d["b"])
    .groupby("grupo")
    .agg(total=("nueva", "sum"))
    .sort_values("total", ascending=False)
)
```

`assign` crea columnas dentro de la cadena. La `lambda` recibe el DataFrame tal como llega a ese paso, así que puedes usar columnas creadas en pasos anteriores. La cadena se lee de arriba abajo como una receta y no deja variables intermedias sueltas.

In [ ]:
pedidos_ej = pd.DataFrame({"cliente": ["A", "B", "A", "C"], "monto": [100.0, 50.0, 70.0, 20.0], "costo": [60.0, 20.0, 50.0, 25.0]})
resumen_ej = (
    pedidos_ej
    .assign(margen=lambda d: d["monto"] - d["costo"])
    .query("margen > 0")
    .groupby("cliente")
    .agg(total_margen=("margen", "sum"))
    .sort_values("total_margen", ascending=False)
)
print(resumen_ej)

### ✍️ Tu turno · Ejercicio 5: reporte de margen por categoría
En **una sola cadena** que empiece en `trimestre`, crea `reporte`:
1. Une con `productos` por producto, quedándote solo con las ventas cuyo producto está en `productos`.
2. Crea `costo` (unidades por costo unitario) y `margen` (importe menos costo).
3. Agrupa por categoría con dos columnas: `total_importe` (suma del importe) y `total_margen` (suma del margen).
4. Crea `margen_pct`: total del margen entre total del importe, por 100, redondeado a 1 decimal.
5. Ordena de mayor a menor `total_margen`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

Copia la estructura del ejemplo: `merge` (con `how="inner"`), `assign`, `groupby`, `agg`, `assign` y `sort_values`.
</details>

<details><summary>💡 Pista 2</summary>

En un mismo `assign` puedes crear `costo` y `margen`: la `lambda` de `margen` ya ve la columna `costo`. El segundo `assign` usa las columnas que dejó `agg`.
</details>

---
## 🏋️ Reto final: el trimestre del banco en dólares
1. `movs`: apila `movs_ene`, `movs_feb` y `movs_mar` con un índice nuevo, y convierte la columna fecha a fecha.
2. Agrega el `segmento` de cada cliente desde `clientes_banco` sin perder movimientos; los clientes que no están en esa tabla quedan con el segmento `"sin registro"`.
3. Agrega el tipo de cambio del día (`tc`) desde `tipos_cambio` (convierte antes su fecha) y crea `monto_usd`: el monto entre el tipo de cambio, con 2 decimales.
4. Agrega `mes`, el mes de la fecha. Las columnas finales de `movs` deben ser: fecha, cliente_id, tipo, monto, segmento, tc, monto_usd y mes.
5. `flujo_segmento_mes`: la suma del monto por segmento y mes, con 2 decimales.
6. `flujo_semanal`: la suma del monto por semana, con 2 decimales.
7. `clientes_inactivos`: una lista ordenada de los clientes de `clientes_banco` que no tuvieron movimientos, y `n_sin_registro`: cuántos movimientos quedaron con segmento `"sin registro"`.

No modifiques las tablas originales.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Después de cada `merge` con `how="left"`, `len(movs)` debe seguir siendo el mismo. Para unir por fecha, las dos columnas deben ser del mismo tipo.
</details>

<details><summary>💡 Pista 2</summary>

Para el tipo de cambio: `tc = tipos_cambio.copy()`, convierte `tc["fecha"]` y haz `movs.merge(tc, on="fecha", how="left")`. `flujo_semanal` necesita la fecha como índice: `movs.set_index("fecha")["monto"].resample("W")`.
</details>

---
## 🚀 Nivel pro (opcional)
1. `origen`: haz `trimestre.merge(clientes, on="cliente_id", how="outer", indicator=True)` y cuenta con `value_counts` cuántas filas hay de cada valor de la columna `_merge` (`both`, `left_only`, `right_only`). Es la forma más rápida de auditar una unión.
2. `ventas_dia_semana`: la suma del importe de `trimestre` por día de la semana (0 es lunes), en una sola cadena que convierta la fecha con `assign`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Apilar varias tablas con `concat` y explicar para qué sirve `ignore_index=True`.
- [ ] Unir dos tablas con `merge` y elegir entre `left`, `inner` y `outer`.
- [ ] Explicar por qué una clave repetida multiplica filas y cómo detectarlo.
- [ ] Encontrar los valores de una tabla que no están en otra.
- [ ] Convertir textos en fechas con el formato correcto y `errors="coerce"`.
- [ ] Extraer mes y día de la semana con `.dt` y calcular días entre fechas.
- [ ] Resumir por mes o por semana con `resample`, y explicar qué pasa con los periodos sin datos.
- [ ] Escribir una cadena de métodos con `assign`.

**Próxima sesión (S13):** integrador de pandas, con un caso bancario de principio a fin y el avance del proyecto.